# PatternBoost: Triangle-free graphs

*MathLearn, University of Galway, June 2026.*

Joshua Maglione with Claude Code Opus 4.7.

---

In this notebook we will reproduce, in miniature, the PatternBoost pipeline from Section 3.3 of the notes. The problem is the classical extremal one: how many edges can a triangle-free graph on $n$ vertices have? The answer is Mantel's theorem (1907), $\lfloor n^2/4 \rfloor$, achieved by the balanced complete bipartite graph. The point of the exercise is *not* to rediscover Mantel — local search alone will hit the bound at the sizes we will work with — but to assemble and observe the full pipeline: score function, local search, encoding, transformer generator, and the closed loop that ties them together.

**Companion sections in the lecture notes.** Sections 3.1 (Encodings), 3.2 (What to learn and how to tell), 3.3 (PatternBoost). The open-ended part also touches Section 3.4 if you choose Track B.

## 0. Setup

In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

torch.manual_seed(0)
np.random.seed(0)

In [ ]:
def plot_graph(G, ax=None, title=None, layout='circular', show_triangles=True):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))

    if layout == 'circular':
        pos = nx.circular_layout(G)
    elif layout == 'spring':
        pos = nx.spring_layout(G, seed=0)
    elif layout == 'kamada_kawai':
        pos = nx.kamada_kawai_layout(G)
    else:
        raise ValueError(f'unknown layout: {layout!r}')

    triangles = [tuple(c) for c in nx.enumerate_all_cliques(G) if len(c) == 3]
    n_edges, n_tri = G.number_of_edges(), len(triangles)

    if show_triangles and triangles:
        for tri in triangles:
            xs = [pos[v][0] for v in tri]
            ys = [pos[v][1] for v in tri]
            ax.fill(xs, ys, color='crimson', alpha=0.18, zorder=1)

    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='gray', width=1.5, alpha=0.7)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color='lightsteelblue',
                           node_size=400, edgecolors='black', linewidths=1)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=10)

    if title is None:
        title = f'|E| = {n_edges},  triangles = {n_tri},  score = {n_edges - 2 * n_tri}'
    ax.set_title(title)
    ax.set_aspect('equal'); ax.axis('off')
    return ax

In [ ]:
# We work at n = 10 throughout. Larger n is suggested in Section 6.
N = 10
M = N * (N - 1) // 2          # number of possible edges 
MANTEL = (N * N) // 4         # correct bound
print(f'n = {N}, sequence length = {M}, Mantel bound = {MANTEL}')

---

## 1. The score function

The PatternBoost score for triangle-free graphs from Section 3.3 of the notes is

$$\mathrm{score}(G) \;=\; |E(G)| \;-\; 2 \cdot t(G),$$

where $t(G)$ is the number of triangles. Triangle-free graphs have $t(G) = 0$, so they score $|E(G)|$ and the maximum score is the extremal edge count. The factor of 2 is a penalty: removing one edge from a triangle drops $|E(G)|$ by 1 but drops $2\,t(G)$ by at least 2, so any locally optimal graph is triangle-free.

In [ ]:
def score_graph(G):
    triangles = sum(nx.triangles(G).values()) // 3
    return G.number_of_edges() - 2 * triangles

Let's look at some sanity checks to make sure our score is correct.

In [ ]:
K4 = nx.complete_graph(4)
plot_graph(K4)
print(f'K_4 score: {score_graph(K4)}')

In [ ]:
K33 = nx.complete_bipartite_graph(3, 3)
plot_graph(K33)
print(f'K_4 score: {score_graph(K33)}')

In [ ]:
G = nx.atlas.graph_atlas(163)
plot_graph(G)
print(f'G score: {score_graph(G)}')

---

## 2. Local search

The local search is the one described in Section 3.3 of the notes. From any graph $G$:
1. **Phase A.** While $G$ has any triangles, remove an edge that participates in the most triangles. This makes $G$ triangle-free.
2. **Phase B.** While there exists a non-edge whose endpoints have no common neighbour, add it. This makes $G$ triangle-saturated.

Phase A guarantees triangle-freeness; phase B fills in extra edges without creating triangles. The final graph has score $|E(G)|$.

### Phase A

In [ ]:
def remove_triangles_greedy(G):
    G = G.copy()
    while sum(nx.triangles(G).values()) > 0:
        # Number of triangles through edge (u, v) = number of common neighbours of u and v
        best_edge, best_count = None, -1
        for u, v in list(G.edges()):
            common = len(set(G.neighbors(u)) & set(G.neighbors(v)))
            if common > best_count:
                best_count, best_edge = common, (u, v)
        G.remove_edge(*best_edge)
    return G

Sanity check:

In [ ]:
result4 = remove_triangles_greedy(K4)
plot_graph(result4)
print(f"K4 score: {score_graph(K4)}")
print(f"After deletion: {score_graph(result4)}")

In [ ]:
result33 = remove_triangles_greedy(K33)
plot_graph(result33)
print(f"K4 score: {score_graph(K33)}")
print(f"After deletion: {score_graph(result33)}")

In [ ]:
result = remove_triangles_greedy(G)
plot_graph(result)
print(f"K4 score: {score_graph(G)}")
print(f"After deletion: {score_graph(G)}")

### Phase B

In [ ]:
def add_edges_greedy(G):
    G = G.copy()
    n = G.number_of_nodes()
    candidates = [(i, j) for i in range(n) for j in range(i + 1, n)
                  if not G.has_edge(i, j)]
    np.random.shuffle(candidates)
    for u, v in candidates:
        if not (set(G.neighbors(u)) & set(G.neighbors(v))):
            G.add_edge(u, v)
    return G

def local_search(G):
    return add_edges_greedy(remove_triangles_greedy(G))

Sanity check:

In [ ]:
K4_ls = local_search(K4)
_ = plot_graph(K4_ls)

In [ ]:
K33_ls = local_search(K33)
_ = plot_graph(K33_ls)

In [ ]:
G_ls = local_search(G)
_ = plot_graph(G_ls)

In [ ]:
G_rand = nx.erdos_renyi_graph(N, 0.5, seed=42)
_ = plot_graph(G_rand)
G_rand_ls = local_search(G_rand)
_ = plot_graph(G_rand_ls)

### 2.3 The local-only baseline

Before we bring in any transformer, let us see how strong local search alone is at this scale. Run it from many random starts and look at the distribution of final scores.

In [ ]:
def local_search_from_random(n_runs=500, n_vertices=N, p_range=(0.2, 0.6)):
    scores = []
    for _ in range(n_runs):
        p = np.random.uniform(*p_range)
        G = nx.erdos_renyi_graph(n_vertices, p)
        G = local_search(G)
        scores.append(score_graph(G))
    return np.array(scores)

scores = local_search_from_random(500)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(scores, bins=range(scores.min(), MANTEL + 2), edgecolor='black', align='left')
ax.axvline(MANTEL, color='crimson', ls='--', label=f'Mantel bound = {MANTEL}')
ax.set_xlabel('score'); ax.set_ylabel('count')
ax.set_title(f'Local-search-from-random scores ({len(scores)} runs)')
ax.legend(); plt.tight_layout(); plt.show()

print(f'best local-only score:  {scores.max()}')
print(f'mean:                   {scores.mean():.2f}')
print(f'fraction hitting bound: {(scores == MANTEL).mean():.2%}')

**Observation.** At $n = 10$ a fair fraction of random starts already reach the Mantel bound after local search. This is important to keep in mind: we are about to add a transformer to this loop, and we should not expect it to *improve on the bound*. What we will see instead is that the transformer learns the *distribution* of high-scoring graphs.

---

## 3. Encoding graphs as token sequences

We need to feed graphs to a transformer. The simplest encoding is the binary upper-triangular adjacency: list the entries $A_{ij}$ for $1 \le i < j \le n$ in some fixed order (we use row-major). This gives a sequence of $\binom{n}{2}$ tokens, each in $\{0, 1\}$. Every such sequence corresponds to a valid graph and vice versa — there is no invalid output to worry about.

This is the encoding the notes discuss in Section 3.1, and it has all the properties you would want: a unique encoding per graph (up to vertex labelling), short, and easy to decode.

In [ ]:
def graph_to_tokens(G, n=N):
    A = nx.to_numpy_array(G, nodelist=range(n), dtype=int)
    i, j = np.triu_indices(n, k=1)
    return torch.tensor(A[i, j], dtype=torch.long)

def tokens_to_graph(tokens, n=N):
    G = nx.empty_graph(n)
    i_idx, j_idx = np.triu_indices(n, k=1)
    edges = [(int(i_idx[k]), int(j_idx[k]))
             for k in range(len(tokens)) if tokens[k].item() == 1]
    G.add_edges_from(edges)
    return G

In [ ]:
G_test = nx.erdos_renyi_graph(N, 0.4, seed=7)
plot_graph(G_test)

In [ ]:
toks = graph_to_tokens(G_test)
toks

In [ ]:
G_back = tokens_to_graph(toks)
plot_graph(G_back)

---

## 4. A tiny transformer generator

We will use a decoder-only character-level transformer over the binary alphabet $\{0, 1\}$ plus a beginning-of-sequence token. The architecture comprises three layers, four heads, 64-dim embeddings — about $10^5$ parameters in total.

This cell defines the model. (Read it but you do not need to edit it; the substance of this notebook is the *loop*, not the architecture.)

In [ ]:
# Out positional vectors
def sinusoidal_pe(n, d):
    pe = torch.zeros(n, d)
    pos = torch.arange(n).float().unsqueeze(1)
    div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

class CausalSelfAttention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        assert d % h == 0
        self.h, self.dk = h, d // h
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
    def forward(self, x):
        B, n, d = x.shape
        qkv = self.qkv(x).view(B, n, 3, self.h, self.dk).permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]
        s = Q @ K.transpose(-2, -1) / math.sqrt(self.dk)
        causal = torch.tril(torch.ones(n, n, device=x.device)).bool()
        s = s.masked_fill(~causal, float('-inf'))
        a = F.softmax(s, dim=-1) @ V
        return self.out(a.transpose(1, 2).contiguous().view(B, n, d))

class CausalBlock(nn.Module):
    def __init__(self, d, h, d_ff):
        super().__init__()
        self.attn = CausalSelfAttention(d, h)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(
            nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d),
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

# Token vocabulary: 0, 1 (edge absent/present), 2 (BOS).
BOS = 2
VOCAB = 3

class GraphGenerator(nn.Module):
    def __init__(self, seq_len=M, d=64, h=4, n_layers=3, d_ff=128):
        super().__init__()
        self.seq_len = seq_len
        self.embed = nn.Embedding(VOCAB, d)
        self.register_buffer('pe', sinusoidal_pe(seq_len + 1, d))
        self.blocks = nn.ModuleList([CausalBlock(d, h, d_ff) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, VOCAB)
    
    def forward(self, x):
        h = self.embed(x) + self.pe[:x.shape[1]].unsqueeze(0)
        for blk in self.blocks:
            h = blk(h)
        return self.head(self.ln(h))
    
    @torch.no_grad()
    def sample(self, batch=128):
        """Autoregressively sample `batch` graph tokenizations from BOS."""
        self.eval()
        x = torch.full((batch, 1), BOS, dtype=torch.long)
        for _ in range(self.seq_len):
            logits = self(x)[:, -1, :2]                 # only allow 0/1 in body
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1)
            x = torch.cat([x, nxt], dim=1)
        return x[:, 1:]                                  # drop BOS

model = GraphGenerator()
print(f'parameters: {sum(p.numel() for p in model.parameters())}')

---

## 5. The PatternBoost loop

We finally close the loop. Algorithmically:

1. Build an initial dataset of high-scoring graphs by running local search on random Erdős–Rényi graphs.
2. Train the generator on this dataset (standard next-token cross-entropy).
3. Sample a batch of new graphs from the trained generator.
4. Run local search on each sample.
5. Merge: keep the top-$K$ across the union of (current dataset, post-search samples).
6. Repeat from 2.

We track three numbers per round:
* mean score of the generator's raw samples (this measures what the model alone knows),
* mean score after local search (this is what we actually get to use),
* mean score of the working dataset (this drives the next round's training).

### 5.1 Initial dataset

In [ ]:
def build_initial_dataset(n_samples=500):
    data, scores = [], []
    for _ in tqdm(range(n_samples), desc='initial LS'):
        p = np.random.uniform(0.2, 0.6)
        G = local_search(nx.erdos_renyi_graph(N, p))
        data.append(graph_to_tokens(G))
        scores.append(score_graph(G))
    return torch.stack(data), np.array(scores)

data, scores = build_initial_dataset(500)
print(f'initial dataset: size = {len(data)}, scores in [{scores.min()}, {scores.max()}], mean = {scores.mean():.2f}')

### 5.2 Training step

In [ ]:
def train_one_round(model, data, epochs=4, batches_per_epoch=50, batch_size=64, lr=5e-4):
    """Standard next-token cross-entropy. Sequences are prefixed with BOS."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    losses = []
    for _ in range(epochs):
        epoch_loss = 0.0
        for _ in range(batches_per_epoch):
            idx = torch.randint(0, data.shape[0], (batch_size,))
            seqs = data[idx]
            bos = torch.full((batch_size, 1), BOS, dtype=torch.long)
            inp = torch.cat([bos, seqs], dim=1)
            x_in, y_target = inp[:, :-1], inp[:, 1:]
            logits = model(x_in)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB), y_target.reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / batches_per_epoch)
    return losses

### 5.3 The loop

In [ ]:
def run_loop(model, data, scores, n_rounds=4, n_samples=128, keep_top=500):
    log = {'raw_mean': [], 'raw_max': [], 'raw_tf_frac': [],
           'ls_mean': [], 'ls_max': [],
           'ds_mean': [], 'ds_max': []}
    
    log['ds_mean'].append(scores.mean())
    log['ds_max'].append(scores.max())
    
    for rnd in range(n_rounds):
        t0 = time.time()
        # Train
        train_one_round(model, data)
        
        # Sample
        samples = model.sample(batch=n_samples)
        raw_graphs = [tokens_to_graph(samples[i]) for i in range(n_samples)]
        raw_scores = np.array([score_graph(g) for g in raw_graphs])
        tf_frac = np.mean([sum(nx.triangles(g).values()) == 0 for g in raw_graphs])
        
        # Local search
        ls_graphs = [local_search(g) for g in raw_graphs]
        ls_scores = np.array([score_graph(g) for g in ls_graphs])
        
        # Update dataset: top-K from union
        new_tokens = torch.stack([graph_to_tokens(g) for g in ls_graphs])
        all_tokens = torch.cat([data, new_tokens], dim=0)
        all_scores = np.concatenate([scores, ls_scores])
        top_idx = np.argsort(all_scores)[-keep_top:]
        data = all_tokens[top_idx]
        scores = all_scores[top_idx]
        
        log['raw_mean'].append(raw_scores.mean()); log['raw_max'].append(raw_scores.max())
        log['raw_tf_frac'].append(tf_frac)
        log['ls_mean'].append(ls_scores.mean()); log['ls_max'].append(ls_scores.max())
        log['ds_mean'].append(scores.mean()); log['ds_max'].append(scores.max())
        
        print(f'Round {rnd+1} ({time.time()-t0:.0f}s): '
              f'raw mean={raw_scores.mean():.1f} TF%={tf_frac:.0%}  '
              f'LS mean={ls_scores.mean():.1f} max={ls_scores.max()}  '
              f'dataset mean={scores.mean():.1f}')
    return data, scores, log

data, scores, log = run_loop(model, data, scores, n_rounds=10)

### 5.4 Visualize what happened

In [ ]:
rounds = np.arange(len(log['raw_mean'])) + 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(rounds, log['raw_mean'], 'o-', label='raw sample mean')
ax.plot(rounds, log['ls_mean'], 's-', label='post-LS mean')
ax.plot(np.arange(len(log['ds_mean'])), log['ds_mean'], '^-', label='dataset mean')
ax.axhline(MANTEL, color='gray', ls='--', label=f'Mantel = {MANTEL}')
ax.set_xlabel('round'); ax.set_ylabel('score')
ax.set_title('Score evolution'); ax.legend(loc='best')

ax = axes[1]
ax.plot(rounds, log['raw_tf_frac'], 'o-')
ax.set_xlabel('round'); ax.set_ylabel('fraction of raw samples triangle-free')
ax.set_title("Model's grasp of the triangle-free constraint")
ax.set_ylim(-0.05, 1.05)

plt.tight_layout(); plt.show()

**Observations to make for yourself.**

* The *dataset mean* should rise monotonically toward the Mantel bound: each round adds graphs that are at least as good as the median, so the working pool drifts upward.
* The *post-LS mean* tracks the dataset mean closely — local search is doing most of the work here.
* The *raw sample mean* is much lower; the generator alone produces many triangles, especially early on. The fraction of raw samples that are triangle-free starts near zero, and is one of the most direct indicators of what the model has learned.

At $n = 10$, local search alone already reaches the bound, so we cannot point to the transformer as the cause of any improvement in the *maximum* score. What we can point to is the shift in the *distribution* of where local search starts: the model learns to propose graphs that, after polishing, are good more reliably than random Erdős–Rényi starts.

### 5.5 What does a sample look like?

Let us pick the best-scoring graph the loop discovered and draw it.

In [ ]:
best_idx = int(np.argmax(scores))
G_best = tokens_to_graph(data[best_idx])
print(f'best score: {scores[best_idx]} (Mantel: {MANTEL})')
print(f'triangle-free: {sum(nx.triangles(G_best).values()) == 0}')
print(f'degree sequence: {sorted([d for _, d in G_best.degree()], reverse=True)}')

# Compare to Turan T(n, 2)
T = nx.complete_bipartite_graph(N // 2, N - N // 2)
print(f'T({N}, 2) degree sequence: {sorted([d for _, d in T.degree()], reverse=True)}')

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
nx.draw_circular(G_best, ax=axes[0], with_labels=True, node_color='lightsteelblue', node_size=400)
axes[0].set_title(f'Best found, score {scores[best_idx]}')
nx.draw_circular(T, ax=axes[1], with_labels=True, node_color='lightcoral', node_size=400)
axes[1].set_title(f'Turán T({N}, 2), score {score_graph(T)}')
plt.tight_layout(); plt.show()

---

## 6. Additional fun

Three suggested experiments, in roughly increasing order of difficulty.

### Track A — A different encoding (30 min)

Re-encode each graph as a list of *edges* (pairs of vertex labels) rather than as an upper-triangular bit string. The vocabulary becomes $\{0, 1, \dots, n-1\}$ plus a SEP/EOS token, and the sequence length is variable (graphs with more edges produce longer sequences).

**Task.**
1. Implement `graph_to_edge_tokens(G)` and `edge_tokens_to_graph(tokens)` with a clear convention (padding strategy, EOS token, etc.).
2. Modify `GraphGenerator` to use this vocabulary and a fixed maximum sequence length.
3. Re-run the loop and compare the score-evolution plot from Section 5 with this encoding.

**Question.** Does the edge-list encoding learn faster or slower than the adjacency encoding? Consider:
* the *sequence length* is shorter for sparse graphs;
* the *constraint* "no two pairs share two vertices and form a triangle" is harder to express locally in the edge list than in the upper-triangular adjacency.

In [ ]:
# TODO: design and implement an edge-list encoder.

# def graph_to_edge_tokens(G, max_len, pad=..., sep=...):
#     ...

# def edge_tokens_to_graph(tokens, n=N):
#     ...

### Track B — $K_4$-free graphs

Mantel's theorem generalizes to Turán's theorem: the maximum number of edges in a $K_{r+1}$-free graph on $n$ vertices is achieved by the balanced complete $r$-partite graph, with edge count $\left(1 - \frac{1}{r}\right) \frac{n^2}{2}$ asymptotically.

For $r = 3$ (so we forbid $K_4$, the complete graph on 4 vertices), the bound at $n = 10$ is $\left(1 - \frac{1}{3}\right) \cdot 50 = 33.3$, achieved by $T(10, 3) = K_{4,3,3}$ with $33$ edges.

**Task.**
1. Modify `score_graph` to penalize $K_4$ subgraphs instead of triangles. (Hint: `nx.find_cliques(G)` or `[c for c in nx.enumerate_all_cliques(G) if len(c) == 4]`.)
2. Modify the two local-search phases accordingly.
3. Re-run the pipeline. Does local search alone reach 33? Does the loop?

**Note.** Counting $K_4$ is slower than counting triangles, so this loop will take a few minutes longer.

In [ ]:
# TODO: define score_K4(G), remove_K4_greedy(G), add_edges_K4_safe(G),
# and call run_loop with these.

# def count_K4(G):
#     return sum(1 for c in nx.enumerate_all_cliques(G) if len(c) == 4)

# def score_K4(G):
#     return G.number_of_edges() - 3 * count_K4(G)
# (3 is the analogue of the factor 2 used for triangles; check why this is the right penalty.)

### Track C — Diagnostics: what did the original model learn?

At $n = 10$, local search by itself reaches the Mantel bound — so the loop's *output* (the max score) is uninformative. But the *raw sample distribution* of the trained generator carries real information about what the model has internalized. Some questions worth asking:

1. **Edge density.** Plot a histogram of $|E(G)|$ for raw samples after each round of training, alongside the histogram of $|E(G)|$ for the working dataset. Do they converge?
2. **Degree distribution.** The Turán graph $T(10, 2) = K_{5,5}$ has a balanced bipartite structure: every vertex has degree exactly 5. Plot the average sorted degree sequence of raw samples over the rounds and compare to the sequence $(5, 5, \dots, 5)$. Does the model approach the bipartite structure?
3. **A more specific test.** For each raw sample, check whether it is bipartite (`nx.is_bipartite`). What fraction are bipartite after training versus at random?

The last question is sharpest: a triangle-free graph need not be bipartite (think of $C_5$), but the *extremal* triangle-free graphs at this $n$ are bipartite. If the model is approaching the bipartite structure, that is non-trivial evidence it has captured something beyond "few triangles".

In [ ]:
# TODO: implement and plot the diagnostics. Suggested starting point:

# def sample_and_diagnose(model, n_samples=256):
#     samples = model.sample(batch=n_samples)
#     graphs = [tokens_to_graph(samples[i]) for i in range(n_samples)]
#     n_edges = np.array([g.number_of_edges() for g in graphs])
#     bip = np.array([nx.is_bipartite(g) for g in graphs])
#     return n_edges, bip, graphs

# Compare these statistics across the rounds you logged in Section 5.

---

## 7. Closing remarks

You have run a complete PatternBoost pipeline. The key lesson might be somewhat anticlimactic at a small scale. The transformer's job is *not* to optimize a hard objective by itself; it is to learn a useful distribution over plausible inputs, which a deterministic search procedure then polishes.

To see the loop genuinely improve over local-only, you need a problem where local search is weaker than at $n = 10$ triangle-free. Two natural next steps:

* Run the same code at $n = 25$ or $n = 30$. The sequence length grows quadratically and so does the training time, but the qualitative story changes: greedy local search no longer hits the bound from random starts, and a learned proposal distribution starts to matter.
* Try the original PatternBoost paper's problem zoo — the planar Turán problem, $f$-vector constraints on polytopes, or any of the others listed in Section 3.3 of the notes. The score function and validity checks change, but the loop structure is exactly what you ran here.